# 3 Loss, optimizer, and the loop

Notebook 01 gave you vectors, notebook 02 gave you the layer. This one closes the gap
between them: how a cache becomes batches, how you tell the model it is wrong, and how it
changes its mind.

None of this is new in 2026 — it is the machinery every course teaches. What is different
is that you already have something worth training on, so we can go through it once,
properly, at speed, and end with a **number that the rest of the lesson has to beat.**

That last part matters more than it sounds. Notebook 04 runs a search over dozens of
architectures. A search without a baseline established first is unreadable — you cannot
tell a win from a wobble.

In [ ]:
import sys
import time
from pathlib import Path

import datasets
import torch
from loguru import logger
from torch import nn
from torch.utils.data import DataLoader
from vectormesh import VectorCache

from notebooktester import param

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()

torch.manual_seed(42)

train_cache = VectorCache.from_hub("pttrn-io/eurosat-dinov2-small", split="train")
test_cache = VectorCache.from_hub("pttrn-io/eurosat-dinov2-small", split="test")

HIDDEN = train_cache.metadata["embed"]["hidden_size"]      # read the metadata, don't guess
N_CLASSES = train_cache.features["label"].num_classes

print(f"{len(train_cache)} train / {len(test_cache)} test rows")
print(f"hidden_size {HIDDEN}, {N_CLASSES} classes")

## 3.1 From cache rows to batches

A cache row is a dict: one vector which is a 1D tensor (,D) and one label which is just the number of the class, eg 5. A model wants a **batch**: one `(B, D)` 2d tensor, which are e.g. with `B=32` there are 32 1D tensors stacked together into a 2D tensor
of inputs and one tensor of `B` targets. The function that turns the first into the second is
called a *collate function*, and it is worth writing once by hand so it stops being magic.

In [ ]:
rows = [train_cache[i] for i in range(4)]
print("one row:", {k: tuple(v.shape) for k, v in rows[0].items()})


def collate_by_hand(rows: list[dict]) -> tuple[torch.Tensor, torch.Tensor]:
    X = torch.stack([row["embed"] for row in rows])          # list of (D) -> (B, D)
    y = torch.stack([row["label"] for row in rows])          # list of ()  -> (B,)
    return X, y


X, y = collate_by_hand(rows)
print(f"\n{len(rows)} rows -> X {tuple(X.shape)}, y {tuple(y.shape)}")

That is the whole job: **a list of items becomes a pair of tensors.** Everything else a
collate function ever does is a variation on it — padding (adding zeros when you need to batch data with different sizes, eg sentences that are sometimes shorter, sometimes longer) when the items have different
lengths, stacking when they do not.

Ours is the easy case. `tensordtype: 1` in the metadata told us each item is `(D)`, so
`torch.stack` is enough. Lesson 5's chunked documents are `(C, D)` where every document is chunked into pieces of 512 tokens, giving a different `C` because the documents have different lengths, and *there* the padder has to do real work.

So use the library version, which is the same idea with the shapes checked:

In [ ]:
from vectormesh.data import Collate, OneHot

# CrossEntropyLoss accepts soft targets, so we one-hot the label into its own column
onehot = OneHot(num_classes=N_CLASSES, label_col="label", target_col="onehot")
collate_fn = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)

BATCH = 64
trainloader = DataLoader(train_cache.dataset.map(onehot), batch_size=BATCH,
                         shuffle=True, collate_fn=collate_fn)
testloader = DataLoader(test_cache.dataset.map(onehot), batch_size=BATCH,
                        shuffle=False, collate_fn=collate_fn)

X, y = next(iter(trainloader))
print(f"X {tuple(X.shape)}  {X.dtype}")
print(f"y {tuple(y.shape)}  {y.dtype}   <- one-hot, so (B, K) not (B,)")
print(f"{len(trainloader)} train batches, {len(testloader)} test batches")

`(B, D) → (B, K)`. That is the entire trip from notebook 01's ladder, and you are now
standing on the second rung.

Note the two loaders differ in one flag. `shuffle=True` on train, because gradient descent
on a fixed order learns the order; `shuffle=False` on test, because there is nothing to
learn there and you want a reproducible number.

## 3.2 Losses: what "wrong" means

A loss is a single number saying how badly the model did, differentiable so that we can ask
which direction reduces it. Picking one is not a detail — it *defines* the task.

| task | one item's target | loss | metric |
|---|---|---|---|
| regression | a real number | `MSELoss` / `L1Loss` | MAE, MASE |
| **single-label** classification | exactly one of K | **`CrossEntropyLoss`** | **Accuracy** |
| multi-label classification | any subset of K | `BCEWithLogitsLoss` | F1 |

**Our task is the middle row.** A EuroSAT tile is a forest or a river, not both, so the ten
scores should compete with each other — which is exactly what the softmax inside
`CrossEntropyLoss` does. Accuracy is the honest metric because the classes are roughly
balanced.

The distinction to hold on to is *single-label vs multi-label*, and it is about the data,
not the model:

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.1])


def show(name, scores, comment):
    print(f"{name:24s} [{', '.join(f'{s:.3f}' for s in scores)}]   sums to {scores.sum():.3f}   {comment}")


show("raw logits", logits, "")
show("softmax (CrossEntropy)", torch.softmax(logits, dim=0), "<- the classes compete")
show("sigmoid (BCEWithLogits)", torch.sigmoid(logits), "<- each class decided alone")

Raising one softmax score pushes the others down; raising one sigmoid score does nothing to
its neighbours. That is the whole difference, and it is why lesson 5's legal-document task
takes `BCEWithLogits` + F1: a ruling can be about tenancy law *and* liability at once, so
forcing those two labels to compete would be a modelling error, not a tuning choice.

Two practical notes that cost people hours:

- **`CrossEntropyLoss` eats logits, not probabilities.** Do not put a softmax at the end of
  your model — the loss applies its own, and doing it twice quietly flattens your gradients.
- **`BCEWithLogits` is `Sigmoid + BCELoss` fused**, and the fused version is the numerically
  stable one. Use it rather than assembling the pieces.

## 3.3 The optimizer, and the loop

Backpropagation gives you $\partial \text{loss} / \partial w$ for every weight: which way to
nudge each one. The optimizer decides how far to step. `Adam` adapts the step size per
parameter and is a fine default for everything in this course.

Here is the loop, by hand, once. Four lines, in this order, and getting the order wrong is
one of the classic silent bugs:

In [ ]:
model = nn.Linear(HIDDEN, N_CLASSES)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(param(3, test=1)):
    total = 0.0
    for X, y in trainloader:
        optimizer.zero_grad()          # 1. clear last batch's gradients -- they ACCUMULATE
        yhat = model(X)                # 2. prediction (forward)
        loss = loss_fn(yhat, y)        # 3. how wrong was the prediction?
        loss.backward()                # 4. gradient of the loss w.r.t. every weight (backward)
        optimizer.step()               # 5. nudge the weights
        total += float(loss.detach())

    model.eval()
    with torch.no_grad():
        correct = sum(float((model(X).argmax(1) == y.argmax(1)).sum()) for X, y in testloader)
    model.train()
    print(f"epoch {epoch}  train loss {total / len(trainloader):.4f}  test acc {correct / len(test_cache):.4f}")

Things worth naming in that loop, because you will meet all of them as bugs:

- **`zero_grad()` first.** Gradients accumulate by default. Forget it and every step uses
  the sum of all gradients so far.
- **`no_grad()` for evaluation.** No gradients are needed, and it halves the memory.
- **`.eval()` / `.train()`.** Some layers behave differently in the two modes — dropout,
  batchnorm, and `GaussianNoise`, which you will meet in notebook 04. Evaluating in train
  mode gives you a noisy, irreproducible number.

That is the last time you will write this loop. It is short enough to understand and
repetitive enough that retyping it per experiment is how mistakes get in. From here on,
`mltrainer.Trainer` runs it, with logging and early stopping attached.

In [ ]:
from mltrainer import ReportTypes, Trainer, TrainerSettings
from vectormesh.components import NeuralNet, Serial
from vectormesh.components.metrics import Accuracy
from vectormesh.data.vectorizers import detect_device

EPOCHS = param(20, test=1)
device = detect_device()


def train(model, trainloader=trainloader, testloader=testloader, epochs=EPOCHS) -> tuple[float, float]:
    settings = TrainerSettings(
        epochs=epochs,
        metrics=[Accuracy()],
        logdir=Path("modellogs").resolve(),
        train_steps=len(trainloader),
        valid_steps=len(testloader),
        reporttypes=[ReportTypes.TENSORBOARD],
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=testloader,
        scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau,
        device=device,
    )
    start = time.time()
    trainer.loop()
    elapsed = time.time() - start

    # loop() logs the metrics but does not return them; one more pass reads the final number
    metrics, _ = trainer.evalbatches()
    return float(metrics["Accuracy"]), elapsed


print(f"training on {device}")

## 3.4 The baseline

Now the number. `NeuralNet(hidden_size, out_size)` is two `Linear` layers with a `GELU`
between them — notebook 02's construction, once. `Serial` chains components; with a single
component it does nothing at all, and it is written this way because notebook 04 will add
things to that list.

This is the model to beat.

In [ ]:
torch.manual_seed(42)
baseline = Serial([NeuralNet(hidden_size=HIDDEN, out_size=N_CLASSES)])

accuracy, secs = train(baseline)
params = sum(p.numel() for p in baseline.parameters() if p.requires_grad)

print(f"\nBASELINE  accuracy {accuracy:.4f}   {params:,} parameters   {secs:.1f}s")

Write that number down. Three things about it:

**It cost seconds.** Not because the problem is easy — because you are training over cached
vectors instead of images. The same accuracy from pixels means a CNN, a GPU, and a coffee
break. That asymmetry is the entire reason this course is arranged the way it is.

**It is already close to the ceiling.** The measured best for a small head on this cache is
around 0.95. Whatever notebook 04's search finds, it is fighting for the last few points,
and you should expect most of its trials to land *below* this line rather than above it.

**It is a two-line model.** No hidden layers you chose, no fusion, no gating, no tuning.
Every architecture you are about to try has to earn the difference — and the honest default
expectation is that most of them will not.

That is not cynicism, it is the reason the search is worth running: you find out, in
seconds, instead of arguing about it. Notebook 04 does exactly that, on a dataset with more
headroom than this one.

## What you now know

- A collate function turns a list of items into `(X, y)`. Stacking is enough for `(D)`
  items; padding is for the chunked case.
- The loss defines the task: `CrossEntropy` when classes compete, `BCEWithLogits` when they
  do not. Feed both of them logits.
- The loop is `zero_grad → forward → loss → backward → step`, and `eval()`/`no_grad()`
  around evaluation.
- You have a baseline on EuroSAT, established *before* any searching. That is what makes
  the next notebook readable.

## Exercises

**E1 — the wrong loss on purpose.** Swap `CrossEntropyLoss` for `BCEWithLogitsLoss` on this
single-label task and retrain. Accuracy barely moves. Explain why the metric fails to
notice, and what would have to be true about EuroSAT for the swap to be *correct* rather
than merely survivable.

**E2 — where does the gap come from?** Shrink the training set to 100 rows
(`train_cache.select(range(100))`) and retrain. Which moves more: the training loss, or the
gap between train and test accuracy? Predict before running, then say which of the two
tells you that you are overfitting.

**E3 — skip a line.** Delete `optimizer.zero_grad()` from the hand-written loop in §3.3 and
run it. Describe what the loss curve does and explain it in terms of accumulated gradients.
(This is the single most common torch bug; meeting it deliberately once is cheaper than
meeting it accidentally at 2am.)

**E4 — is your baseline stable?** Rerun §3.4 with three different seeds and report the
spread. You now know how big a "win" has to be in notebook 04 before it means anything.